# 1. Create a database connection 🔌🏦

In [18]:
import pandas as pd
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

In [19]:
# 1. Set up connection (update credentials and host!)
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False)



In [35]:
my_schema = "team_3"


mart_customers = pd.read_sql(f"SELECT * FROM {my_schema}.mart_customers;", engine)
mart_discount_coupon = pd.read_sql(f"SELECT * FROM {my_schema}.mart_discount_coupon;", engine)
mart_holidays_2019_us = pd.read_sql(f"SELECT * FROM {my_schema}.mart_holidays_2019_us;", engine)
mart_marketing_spend = pd.read_sql(f"SELECT * FROM {my_schema}.mart_marketing_spend;", engine)
mart_online_sales = pd.read_sql(f"SELECT * FROM {my_schema}.mart_online_sales;", engine)
mart_tax_amount = pd.read_sql(f"SELECT * FROM {my_schema}.mart_tax_amount;", engine)


1. CAC (Customer Acquisition Cost)

   CAC = Total Marketing Spend in Period / Number of New Customers Acquired in Same Period

In [36]:
print(mart_customers.columns)
print(mart_customers.head())

Index(['customer_id', 'gender', 'location', 'tenure_months'], dtype='object')
   customer_id gender    location  tenure_months
0        17850      M    Illinois             12
1        13047      M  California             43
2        12583      M    Illinois             33
3        13748      F  California             30
4        15100      M  California             49


In [37]:
print(mart_online_sales.columns)
print(mart_online_sales.head())

Index(['customer_id', 'transaction_id', 'transaction_date', 'produkt_sku',
       'product_category', 'quantity', 'avg_price', 'delivery_charges',
       'coupon_status', 'revenue', 'transaction_month',
       'transaction_month_str', 'transaction_day_of_month',
       'transaction_day_of_week'],
      dtype='object')
   customer_id  transaction_id transaction_date     produkt_sku  \
0        17850           16679       2019-01-01  GGOENEBJ079499   
1        17850           16680       2019-01-01  GGOENEBJ079499   
2        17850           16681       2019-01-01  GGOEGFKQ020399   
3        17850           16682       2019-01-01  GGOEGAAB010516   
4        17850           16682       2019-01-01  GGOEGBJL013999   

  product_category  quantity  avg_price  delivery_charges coupon_status  \
0         Nest-USA         1     153.71               6.5          Used   
1         Nest-USA         1     153.71               6.5          Used   
2           Office         1       2.05             

In [46]:
print(mart_marketing_spend.columns)
print(mart_marketing_spend.head())

Index(['date', 'offline_spend', 'online_spend', 'month'], dtype='object')
        date  offline_spend  online_spend    month
0 2019-01-01           4500       2424.50  2019-01
1 2019-01-02           4500       3480.36  2019-01
2 2019-01-03           4500       1576.38  2019-01
3 2019-01-04           4500       2928.55  2019-01
4 2019-01-05           4500       4055.30  2019-01


In [40]:
# Make sure transaction_date is a datetime
mart_online_sales['transaction_date'] = pd.to_datetime(mart_online_sales['transaction_date'])

# Find each customer's first purchase month
first_orders = mart_online_sales.groupby('customer_id')['transaction_date'].min().reset_index()
first_orders['signup_month'] = first_orders['transaction_date'].dt.to_period('M')


In [42]:
#Step 2: Count New Customers Per Month

new_customers = first_orders.groupby('signup_month')['customer_id'].nunique().reset_index(name='new_customers')


In [47]:
#Step 3: Calculate Monthly Marketing Spend
# Ensure date is datetime (if not already)
mart_marketing_spend['date'] = pd.to_datetime(mart_marketing_spend['date'])
mart_marketing_spend['month'] = mart_marketing_spend['date'].dt.to_period('M')

# Calculate total marketing spend as the sum of both columns
mart_marketing_spend['total_spend'] = (
    mart_marketing_spend['offline_spend'].fillna(0) + 
    mart_marketing_spend['online_spend'].fillna(0)
)

# Group by month to get monthly total spend
monthly_spend = mart_marketing_spend.groupby('month')['total_spend'].sum().reset_index()

In [ ]:
# 4. Merge and calculate CAC
cac = pd.merge(new_customers, monthly_spend, left_on='signup_month', right_on='month', how='inner')
cac['CAC'] = cac['total_spend'] / cac['new_customers']

print(cac[['signup_month', 'new_customers', 'total_spend', 'CAC']])

   signup_month  new_customers  total_spend          CAC
0       2019-01            215    154928.95   720.599767
1       2019-02             96    137107.92  1428.207500
2       2019-03            177    122250.09   690.678475
3       2019-04            163    157026.83   963.354785
4       2019-05            112    118259.64  1055.889643
5       2019-06            137    134318.14   980.424380
6       2019-07             94    120217.85  1278.913298
7       2019-08            135    142904.15  1058.549259
8       2019-09             78    135514.54  1737.365897
9       2019-10             87    151224.65  1738.214368
10      2019-11             68    161144.96  2369.778824
11      2019-12            106    198648.75  1874.044811
